In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import gc
import warnings
import numpy as np
import pickle
import time
start=time.time()
warnings.filterwarnings("ignore")

In [50]:
calender=pd.read_csv(r"Data\calendar.csv")

In [51]:
sellprices=pd.read_csv(r"Data/sell_prices.csv")

In [52]:
sales=pd.read_csv(r"Data\sales_train_evaluation.csv")

In [53]:
def save_mem(sales, verbose=True):
    print("BEFORE ",sales.memory_usage(deep=True).sum() / 1024**2, "MB")
    for i in sales.columns:
        if sales[i].dtype=="int64":
            sales[i]=pd.to_numeric(sales[i],downcast="integer")
        elif sales[i].dtype=="float64":
            sales[i]=pd.to_numeric(sales[i],downcast="float")
        elif sales[i].dtype=='object':
                if i=="sales":
                     continue
                # convert to category IF it has repeated/limited values
                num_unique = sales[i].nunique()
                num_total = len(sales[i])
                if num_unique / num_total < 0.5:  # heuristic: repeats a lot
                    sales[i] = sales[i].astype('category')
            
    gc.collect()
    print("AFTER ",sales.memory_usage(deep=True).sum() / 1024**2, "MB")
    return sales

In [54]:
calender.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [55]:
calender.drop("weekday",axis=1,inplace=True)

In [56]:
calender[["event_name_1","event_type_1","event_name_2","event_type_2"]].isnull().sum()

event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
dtype: int64

In [57]:
calender[["event_name_1","event_type_1","event_name_2","event_type_2"]]=calender[["event_name_1","event_type_1","event_name_2","event_type_2"]].fillna("None")
calender[["event_name_1","event_type_1","event_name_2","event_type_2"]].isnull().sum()

event_name_1    0
event_type_1    0
event_name_2    0
event_type_2    0
dtype: int64

In [58]:
calender=save_mem(calender)

BEFORE  0.718256950378418 MB
AFTER  0.24149131774902344 MB


In [59]:
calender['date'] = pd.to_datetime(calender['date'])
calender['day_month'] = calender['date'].dt.day
calender['week_month'] = np.ceil(calender['day_month'] / 7) ## np.ceil is used for rounding off 
calender["week_end"]=calender["wday"].isin([1,2])
gc.collect()

0

In [60]:
dupes = sellprices.duplicated(subset=['store_id','item_id','wm_yr_wk']).sum()
print(dupes)# CHECKS WETHER WITHIN A WEEK A PRICE CHANGE HAD OCCURED
dupes = calender.duplicated(subset=["date"]).sum()
print(dupes)
del dupes
gc.collect()

0
0


0

In [61]:
sellprices['price_max'] = sellprices.groupby(['store_id','item_id'])['sell_price'].transform('max')
sellprices['price_min'] = sellprices.groupby(['store_id','item_id'])['sell_price'].transform('min')
#sellprices['price_std'] = sellprices.groupby(['store_id','item_id'])['sell_price'].transform('std') price std is flagged bcz of permutation importance
sellprices['price_mean'] = sellprices.groupby(['store_id','item_id'])['sell_price'].transform('mean')
sellprices['price_norm'] = sellprices['sell_price']/sellprices['price_max']

In [62]:
sellprices=save_mem(sellprices)

BEFORE  1061.9069395065308 MB
AFTER  163.34851551055908 MB


In [63]:
gc.collect()
sellprices

,store_id,item_id,wm_yr_wk,sell_price,price_max,price_min,price_mean,price_norm
0,CA_1,HOBBIES_1_001,11325,9.58,9.58,8.26,8.285714,1.000000
1,CA_1,HOBBIES_1_001,11326,9.58,9.58,8.26,8.285714,1.000000
2,CA_1,HOBBIES_1_001,11327,8.26,9.58,8.26,8.285714,0.862213
3,CA_1,HOBBIES_1_001,11328,8.26,9.58,8.26,8.285714,0.862213
4,CA_1,HOBBIES_1_001,11329,8.26,9.58,8.26,8.285714,0.862213
...,...,...,...,...,...,...,...,...
6841116,WI_3,FOODS_3_827,11617,1.00,1.00,1.00,1.000000,1.000000
6841117,WI_3,FOODS_3_827,11618,1.00,1.00,1.00,1.000000,1.000000
6841118,WI_3,FOODS_3_827,11619,1.00,1.00,1.00,1.000000,1.000000
6841119,WI_3,FOODS_3_827,11620,1.00,1.00,1.00,1.000000,1.000000


In [64]:
# 'id_vars' are the columns you want to keep 
# 'var_name' is what you want to name the new column containing the dates
# 'value_name' is what you want to name the column containing the data under those dates
sales=sales.melt(id_vars=["item_id","dept_id","cat_id","store_id","state_id"],var_name="Date",value_name="sales")
### TAKES LONG TIME
sales=save_mem(sales)

BEFORE  20875.40807914734 MB
AFTER  2486.239086151123 MB


In [65]:
sales=pd.merge(sales,calender,left_on=["Date"],right_on=["d"])
sales.drop(["d","date"],inplace=True,axis=1)
sales=save_mem(sales)
gc.collect()

BEFORE  6910.116503715515 MB
AFTER  3725.418433189392 MB


0

In [66]:
print(len(sellprices))

6841121


In [67]:
cal_week = calender[['wm_yr_wk', 'month','year']].drop_duplicates('wm_yr_wk')
sellprices = sellprices.merge(cal_week,on='wm_yr_wk',how='left',validate='many_to_one')
del calender

In [68]:
sellprices['price_lag_w1']  = sellprices.groupby(['store_id','item_id'])['sell_price'].shift(1)   # 1 week ago
#sellprices['price_lag_m1']  = sellprices.groupby(['store_id','item_id'])['sell_price'].shift(5)   # 1 month FLAGGED DUE TO PERMUATION IMPORTANCE
sellprices['price_lag_m6'] = sellprices.groupby(['store_id','item_id'])['sell_price'].shift(26)    # 6 month ago
sellprices=save_mem(sellprices)
sellprices['price_momentum_w1']  = sellprices['sell_price'] / sellprices['price_lag_w1']
#sellprices['price_momentum_m1'] = sellprices['sell_price'] / sellprices['price_lag_m1'] # 1 month FLAGGED DUE TO PERMUATION IMPORTANCE
sellprices['price_momentum_m6'] = sellprices['sell_price'] / sellprices['price_lag_m6']
sellprices=save_mem(sellprices)
"""price_max — used for rel_price (current/max), your discount-detection signal
 price_mean — the item's typical/average price level
price_std — how much this item's price fluctuates — an item with high price_std is promotion-heavy/volatile; low price_std means it's basically always the same price"""

BEFORE  235.11473083496094 MB
AFTER  235.11473083496094 MB
BEFORE  287.30834197998047 MB
AFTER  287.30834197998047 MB


"price_max — used for rel_price (current/max), your discount-detection signal\n price_mean — the item's typical/average price level\nprice_std — how much this item's price fluctuates — an item with high price_std is promotion-heavy/volatile; low price_std means it's basically always the same price"

In [69]:
sample = sellprices[(sellprices['store_id'] == 'CA_1') & (sellprices['item_id'] == 'HOBBIES_1_001')].sort_values('wm_yr_wk')

In [70]:
sales_sorted = sellprices.sort_values(['item_id', 'store_id', 'wm_yr_wk']) 
newlaunch_sorted = (sales_sorted["sell_price"].notna().groupby([sales_sorted["item_id"], sales_sorted["store_id"]]).cumsum() > 0)
print(len(sales_sorted[newlaunch_sorted]))
print(len(sales_sorted[sales_sorted["sell_price"].notna()])) 
### USED TO FID WHETHER ANY PRODUCT HAD NAN VALUES IN BETWEEN THE TRAINING DATA BUT NON FOUND AS CUSUM AND NOTNA PRODUCES SALES LENGTH
del sales_sorted,newlaunch_sorted
gc.collect()

6841121
6841121


0

In [71]:
gc.collect()
sales=pd.merge(sellprices.drop(["year","month"],axis=1),sales,on=["item_id","store_id","wm_yr_wk"],how="right")
# WHY HOW=right JOINS EVERY SALES ROW IF EVEN A PRICE IS MISSING AT THAT POINT THE PRODUCTS
#  WHICH ARE NOT AVAILABLE DURING TAHT TIME IS RECOREDED AS NAN
## THE INDEX CANT BE THE DATE COLUMN AS THE DATE IN THE ABOVE SALES DATA GETS REPEATED
del sellprices
gc.collect()

0

In [72]:
sales=sales.dropna(subset=["sell_price"])


In [73]:
sales["sell_price"].isna().sum()

np.int64(0)

In [74]:
gc.collect()
rmsse={}
date=1941
sales=sales[sales["sell_price"].isna()==False]

In [75]:
sales["Date"] = sales["Date"].str.replace("d_", "", regex=False)
sales["Date"]=sales["Date"].astype("int16")


In [76]:
lookups = {}
lookups["dept_id"]  = sales[sales["Date"]<=date].groupby("dept_id")["sales"].agg(["mean"]) #Flagged std due to permutation importance
lookups["item_id"]  = sales[sales["Date"]<=date].groupby("item_id")["sales"].agg(["mean","std"])
lookups["cat_id"]   = sales[sales["Date"]<=date].groupby("cat_id")["sales"].agg(["std"]) #Flagged std due to permutation importance
lookups["state_id"] = sales[sales["Date"]<=date].groupby("state_id")["sales"].agg(["mean","std"])
lookups["state_dept_id"]   = sales[sales["Date"]<=date].groupby(["state_id","dept_id"])["sales"].agg(["std"])
lookups["state_item_id"]   = sales[sales["Date"]<=date].groupby(["state_id","item_id"])["sales"].agg(["mean"])
lookups["store_id"]=sales[sales["Date"]<=date].groupby("store_id")["sales"].agg(["mean","std"])
for name in lookups:
    if name in ("cat_id","state_dept_id"):
        lookups[name].columns = [f"{name}_enc_std"]
        lookups[name] = lookups[name].reset_index()
        continue
    if name in ("dept_id","state_item_id"):
        lookups[name].columns = [f"{name}_enc_mean"]
        lookups[name] = lookups[name].reset_index()
        continue
    lookups[name].columns = [f"{name}_enc_mean", f"{name}_enc_std"]
    lookups[name] = lookups[name].reset_index()

In [77]:
group_map = {
    "dept_id":        "dept_id",
    "item_id":        "item_id",
    "state_item_id":["state_id","item_id"],
    "cat_id":         "cat_id",
    "state_dept_id":   ["state_id","dept_id"],
}


In [78]:
OTHER=r"D:\M5 Forecasting"

In [79]:
with open(OTHER + r"\Sale-Store\lookups.pkl", "wb") as f:
    pickle.dump({"lookups": lookups, "group_map": group_map}, f)
print("Saved lookups + group_map ->", OTHER + r"\Sale-Store\lookups.pkl")

Saved lookups + group_map -> D:\M5 Forecasting\Sale-Store\lookups.pkl


In [80]:
## WHY NOT DROP NA IN THE ROLLING FEATURES OR ANY OTHER EXYTA FEATURES CREATUED COLUMNS IS BCZ WHEN THE ITEM WAS INTRODUCED
#  WEEK WEEKS BEFORE THE 1941 DAYS THE ITEM WILL END UP GETTING COMPLETYL NAN VALUES IF IT WAS INTRODUCED IN THE LESS THAT 26 weeks before the prediction window

In [81]:
def compute_rmsse_scale(train_df):
    train_df = train_df.sort_values("Date")
    scales = {}
    for item, g in train_df.groupby("item_id"):
        vals = g["sales"].to_numpy()
        nz = np.flatnonzero(vals)
        if len(nz) == 0:
            scales[item] = np.nan
            continue
        vals = vals[nz[0]:]          # RMSSE definition: start from first non-zero demand
        if len(vals) < 2:
            scales[item] = np.nan
            continue
        scales[item] = np.mean(np.diff(vals) ** 2)
    return pd.Series(scales, name="scale")

In [82]:
featurestodrop=['wm_yr_wk','dept_id','state_id','cat_id']
sales


,store_id,item_id,wm_yr_wk,sell_price,price_max,price_min,price_mean,price_norm,price_lag_w1,price_lag_m6,...,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,day_month,week_month,week_end
7,CA_1,HOBBIES_1_008,11101,0.46,0.50,0.42,0.476312,0.920000,NaN,NaN,...,None,None,None,None,0,0,0,29,5.0,True
8,CA_1,HOBBIES_1_009,11101,1.56,1.77,1.56,1.764787,0.881356,NaN,NaN,...,None,None,None,None,0,0,0,29,5.0,True
9,CA_1,HOBBIES_1_010,11101,3.17,3.17,2.97,2.981348,1.000000,NaN,NaN,...,None,None,None,None,0,0,0,29,5.0,True
11,CA_1,HOBBIES_1_012,11101,5.98,6.52,5.98,6.469503,0.917178,NaN,NaN,...,None,None,None,None,0,0,0,29,5.0,True
14,CA_1,HOBBIES_1_015,11101,0.70,0.72,0.68,0.706596,0.972222,NaN,NaN,...,None,None,None,None,0,0,0,29,5.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59181085,WI_3,FOODS_3_823,11617,2.98,2.98,2.48,2.801560,1.000000,2.98,2.50,...,None,None,None,None,0,0,0,22,4.0,True
59181086,WI_3,FOODS_3_824,11617,2.48,2.68,2.00,2.507979,0.925373,2.48,2.00,...,None,None,None,None,0,0,0,22,4.0,True
59181087,WI_3,FOODS_3_825,11617,3.98,4.38,3.98,4.115957,0.908676,3.98,3.98,...,None,None,None,None,0,0,0,22,4.0,True
59181088,WI_3,FOODS_3_826,11617,1.28,1.28,1.28,1.280000,1.000000,1.28,1.28,...,None,None,None,None,0,0,0,22,4.0,True


In [83]:
## DROPPING THE ROWS WHERE THE PRODUCTS IS NOT RELEASED
gc.collect()
sales=sales[sales["sell_price"].isna()==False]
#INSIGHT FROM THE DATA GRAPH IN THE DATA ANLYSIS NOTEBOOK
## ALL THE STATES HAVE SIMILAR DEPARTMENT SALES AND HIGHER SALES IS FOUND IN THE FOOD_3 CATEGORY WITH LEAST IN HOBBIES_2 AND HOUSEHOLD_2 
# SO LETS DO MEAN ENCODING FOR THE FEATURE DEPT_ID WITH MEAN ENCODING 
# LABEL ENCODING WITH ORDER IS SIMILAR TO MEAN ENCODING IN TREE BASED MODELS    
"""
.mean() shrinks your data down to one row per group.

.transform("mean") keeps your data the esalesact same size, broadcasting the group's answer back to every individual row"""
## THE TREND IN THE DIFF STORES ACROSS YEARS IS CHANGING 
### THE YEAR DOES NOT EVEN SHOW SIMILAR BEHEVIOUR ACROSS THE STATE ITSELF
### THE YEAR TO BE ENCODED TO BE PROCESSED DIFF FOR EACH STORE AND LETS LEAVE IT TO THE MODEL AS WE ARE GOING TO USE ANY TREE MODEL 
### AND TREE MOEL WILL EVENTUALLY FIND A BEST SPLIT CROSS YEARS
## LETS TRY DIFF GROUPS AND MENA ENCODE IT 

for i in sales['store_id'].unique():
    print("BUILD DATA ",i)
    x=sales[sales["store_id"]==i]
    if i[0:2]=="CA":
        x.drop(["snap_TX","snap_WI"],axis=1,inplace=True)
    elif i[0:2]=="TX":
        x.drop(["snap_CA","snap_WI"],axis=1,inplace=True)
    else:
        x.drop(["snap_CA","snap_TX"],axis=1,inplace=True)
    for name, cols in group_map.items():
        x = x.merge(lookups[name], on=cols, how="left")
    ### WE USE THE ROLLING MEAN TO COMPUTE THE LONG TERM TREND OF THE ITEM
    shift=28
    g = x.groupby(["item_id"])["sales"]
    x["rolling_sold_mean_7"] = (g.transform(lambda x: x.shift(28).rolling(7).mean()))
    x["rolling_sold_std_7"] = (g.transform(lambda x: x.shift(28).rolling(7).std()))
    x["rolling_sold_mean_14"] = (g.transform(lambda x: x.shift(28).rolling(14).mean()))
    x["rolling_sold_std_14"] = (g.transform(lambda x: x.shift(28).rolling(14).std()))
    x["rolling_sold_mean_28"] = (g.transform(lambda x: x.shift(28).rolling(28).mean()))
    #x["rolling_sold_std_28"] = (g.transform(lambda x: x.shift(28).rolling(28).std())) #FOUND WRONG BY PERMUATION TEST
    x["rolling_sold_mean_60"] = (g.transform(lambda x: x.shift(28).rolling(60).mean()))
    #x["rolling_sold_std_60"] = (g.transform(lambda x: x.shift(28).rolling(60).std()))## LONG TERM TREND
    lag=[1,2,7,14,30] #THE 14 IS REMOVED FORM IT BCZ OF  THE PERMUTATION
    for j in lag:
        x[f"sales_lag_{j+28}"]=x.groupby(["item_id"])["sales"].shift(j+28)
    x["sellingTrend"] = x["sales_lag_29"] - x["item_id_enc_mean"]    
    VAL_HORIZON = 28
    
    x[(x["Date"] > date) & (x["Date"] <= date + VAL_HORIZON)].drop(featurestodrop,axis=1).drop("sales",axis=1).to_pickle(OTHER+fr"\Predict-Store\sales-{i}.pkl")
     ## WE CREATE ROLLING FEATURES ACROSS DIFF SHIFTS 
    g = x.groupby(["item_id"])["sales"]
    for d_shift in [1, 7]:
        print("Shifting period:", d_shift)
        for d_window in [7, 14, 30]:
            col_name = f"rolling_mean_{d_shift}_{d_window}"
            x[col_name] = (g.shift(d_shift).groupby([x["item_id"]]).rolling(d_window).mean().droplevel(00))
    x[x["Date"]<=date].drop(featurestodrop,axis=1).to_pickle(OTHER+fr"\Sale-Store\sales-{i}.pkl")
    x[(x["Date"] > date) & (x["Date"] <= date + VAL_HORIZON)][["item_id", "store_id", "Date", "sales"]].to_pickle(OTHER+fr"\Predict-Store\sales-correct-{i}.pkl")  
    scale = compute_rmsse_scale(x[x["Date"]<=date])
    rmsse[i]=scale
    del x

BUILD DATA  CA_1
Shifting period: 1
Shifting period: 7
BUILD DATA  CA_2
Shifting period: 1
Shifting period: 7
BUILD DATA  CA_3
Shifting period: 1
Shifting period: 7
BUILD DATA  CA_4
Shifting period: 1
Shifting period: 7
BUILD DATA  TX_1
Shifting period: 1
Shifting period: 7
BUILD DATA  TX_2
Shifting period: 1
Shifting period: 7
BUILD DATA  TX_3
Shifting period: 1
Shifting period: 7
BUILD DATA  WI_1
Shifting period: 1
Shifting period: 7
BUILD DATA  WI_2
Shifting period: 1
Shifting period: 7
BUILD DATA  WI_3
Shifting period: 1
Shifting period: 7


In [84]:
##FUTURE SKELETON
HORIZON = 28
HIST_BUFFER = 100   
future_days = list(range(date + 1, date + HORIZON + 1))
if sales["Date"].max() < future_days[-1]:
    print(f"'sales' only has real data through day {sales['Date'].max()} — building skeleton for {future_days[0]}-{future_days[-1]}")
    item_store_combos = sales[["item_id","dept_id","cat_id","store_id","state_id"]].drop_duplicates()
    future_skeleton = item_store_combos.merge(pd.DataFrame({"Date": future_days}), how="cross")
    future_skeleton["sales"] = np.nan
    # attach calendar for the future days — calendar.csv already covers this range
    cal_raw = pd.read_csv(r"Data\calendar.csv")
    cal_raw.drop("weekday", axis=1, inplace=True)
    cal_raw[["event_name_1","event_type_1","event_name_2","event_type_2"]] = (cal_raw[["event_name_1","event_type_1","event_name_2","event_type_2"]].fillna("None"))
    cal_raw['date'] = pd.to_datetime(cal_raw['date'])
    cal_raw['day_month'] = cal_raw['date'].dt.day
    cal_raw['week_month'] = np.ceil(cal_raw['day_month'] / 7)
    cal_raw["week_end"] = cal_raw["wday"].isin([1,2])
    cal_raw["Date"] = cal_raw["d"].str.replace("d_", "", regex=False).astype("int16")
    future_skeleton = future_skeleton.merge(cal_raw.drop(columns=["d","date"]), on="Date", how="left")
    # attach price features via wm_yr_wk — reuse price columns already computed on `sales` itself
    price_lookup = sales[["item_id","store_id","wm_yr_wk","sell_price","price_max","price_min",
                            "price_mean","price_norm","price_lag_w1","price_lag_m6",
                            "price_momentum_w1","price_momentum_m6"]].drop_duplicates(
                            subset=["item_id","store_id","wm_yr_wk"])
    future_skeleton = future_skeleton.merge(price_lookup, on=["item_id","store_id","wm_yr_wk"], how="left")

    # keep only enough real history to safely feed shift28+window60, then append the future rows
    hist = sales[sales["Date"] > date - HIST_BUFFER].copy()
    sales = pd.concat([hist, future_skeleton], ignore_index=True)
    del item_store_combos, future_skeleton, cal_raw, price_lookup, hist
    gc.collect()
    print("sales now has", len(sales), "rows, up to day", sales["Date"].max())
else:
    print("'sales' already has real data through the target window — skipping skeleton build")  

'sales' only has real data through day 1941 — building skeleton for 1942-1969
sales now has 3902720 rows, up to day 1969


In [85]:
'''Baseline RMSE (CA_1): 2.0405
               feature  rmse_increase
      rolling_mean_1_7       1.091609
     rolling_mean_1_14       0.411966
      rolling_mean_7_7       0.134501
                  wday       0.100151
state_item_id_enc_mean       0.064754
     rolling_mean_1_30       0.022510
              week_end       0.016773
     rolling_mean_7_14       0.014831
      dept_id_enc_mean       0.005550
    rolling_mean_14_14       0.004157
     rolling_mean_7_30       0.003934
    rolling_mean_14_30       0.003464
  rolling_sold_mean_60       0.003292
             price_min       0.002886
    rolling_sold_std_7       0.002082
  rolling_sold_mean_14       0.001915
     rolling_mean_14_7       0.001492
            sell_price       0.001307
          sales_lag_30       0.001084
          price_lag_w1       0.001020
             price_max       0.000853
  rolling_sold_mean_28       0.000685
          price_lag_m6       0.000673
     price_momentum_w1       0.000652
          sales_lag_58       0.000503
   rolling_sold_std_14       0.000474
                 month       0.000322
             day_month       0.000273
        cat_id_enc_std       0.000035
               snap_CA       0.000024
                  year       0.000000
                  Date       0.000000
            week_month      -0.000033
            price_norm      -0.000088
          sellingTrend      -0.000195
            price_mean      -0.000244
     price_momentum_m6      -0.000413
 state_dept_id_enc_std      -0.000682
       item_id_enc_std      -0.000940
 state_item_id_enc_std      -0.002393
          sales_lag_35      -0.002638
   rolling_sold_std_60      -0.002962
      item_id_enc_mean      -0.003450
state_dept_id_enc_mean      -0.003777
             price_std      -0.004063
       cat_id_enc_mean      -0.004690
          sales_lag_42      -0.005209'''

'Baseline RMSE (CA_1): 2.0405\n               feature  rmse_increase\n      rolling_mean_1_7       1.091609\n     rolling_mean_1_14       0.411966\n      rolling_mean_7_7       0.134501\n                  wday       0.100151\nstate_item_id_enc_mean       0.064754\n     rolling_mean_1_30       0.022510\n              week_end       0.016773\n     rolling_mean_7_14       0.014831\n      dept_id_enc_mean       0.005550\n    rolling_mean_14_14       0.004157\n     rolling_mean_7_30       0.003934\n    rolling_mean_14_30       0.003464\n  rolling_sold_mean_60       0.003292\n             price_min       0.002886\n    rolling_sold_std_7       0.002082\n  rolling_sold_mean_14       0.001915\n     rolling_mean_14_7       0.001492\n            sell_price       0.001307\n          sales_lag_30       0.001084\n          price_lag_w1       0.001020\n             price_max       0.000853\n  rolling_sold_mean_28       0.000685\n          price_lag_m6       0.000673\n     price_momentum_w1       0.00

In [86]:
import pickle
with open(OTHER+r"\Sale-Store\rmsselookup.pkl", "wb") as f:
    pickle.dump(rmsse, f)

In [87]:
#x[x["sell_price"].isna()!=True].groupby("item_id")["wm_yr_wk"].agg("min").max()-11101
## MEANS AT THE MAX A PRODUCT WAS INTRODUCED AFTER 502 WEEKS 

In [88]:
end=time.time()
print("Time Taken ",end-start)

Time Taken  605.9210059642792


In [89]:
# ============================================================
# BUILD STATIC FUTURE SKELETON — saved separately, never touches Predict-Store
# Change `date` here and rerun this cell to target a different window
# ============================================================
import os
VAL_HORIZON = 28
HIST_BUFFER = 100   # covers shift28+window60 = 88 days needed, with margin

FUTURE_DIR = OTHER + r"\Predict-Store-Future"
os.makedirs(FUTURE_DIR, exist_ok=True)

future_days = list(range(date + 1, date + VAL_HORIZON + 1))

if sales["Date"].max() < future_days[-1]:
    print(f"Building future skeleton for days {future_days[0]}-{future_days[-1]}")

    # 1. item/store/dept/cat/state skeleton, one row per future day
    item_store_combos = sales[["item_id","dept_id","cat_id","store_id","state_id"]].drop_duplicates()
    future_skeleton = item_store_combos.merge(pd.DataFrame({"Date": future_days}), how="cross")
    future_skeleton["sales"] = np.nan

    # 2. calendar — calendar.csv already covers this range
    cal_raw = pd.read_csv(r"Data\calendar.csv")
    cal_raw.drop("weekday", axis=1, inplace=True)
    cal_raw[["event_name_1","event_type_1","event_name_2","event_type_2"]] = (
        cal_raw[["event_name_1","event_type_1","event_name_2","event_type_2"]].fillna("None")
    )
    cal_raw['date'] = pd.to_datetime(cal_raw['date'])
    cal_raw['day_month'] = cal_raw['date'].dt.day
    cal_raw['week_month'] = np.ceil(cal_raw['day_month'] / 7)
    cal_raw["week_end"] = cal_raw["wday"].isin([1,2])
    cal_raw["Date"] = cal_raw["d"].str.replace("d_", "", regex=False).astype("int16")
    future_skeleton = future_skeleton.merge(cal_raw.drop(columns=["d","date"]), on="Date", how="left")

    # 3. price — reuse price columns already computed on `sales`, matched via wm_yr_wk
    price_lookup = sales[["item_id","store_id","wm_yr_wk","sell_price","price_max","price_min",
                            "price_mean","price_norm","price_lag_w1","price_lag_m6",
                            "price_momentum_w1","price_momentum_m6"]].drop_duplicates(
                            subset=["item_id","store_id","wm_yr_wk"])
    future_skeleton = future_skeleton.merge(price_lookup, on=["item_id","store_id","wm_yr_wk"], how="left")

    # 4. combine with enough real history to feed shift28+window60
    hist = sales[sales["Date"] > date - HIST_BUFFER].copy()
    combined = pd.concat([hist, future_skeleton], ignore_index=True)

    del item_store_combos, future_skeleton, cal_raw, price_lookup, hist
    gc.collect()

    # 5. per-store: mean encodings + STATIC rolling/lag features only (no dynamic block here)
    for i in combined["store_id"].unique():
        print("BUILD FUTURE STATIC DATA:", i)
        x = combined[combined["store_id"] == i].copy()

        if i[0:2] == "CA":
            x.drop(["snap_TX","snap_WI"], axis=1, inplace=True)
        elif i[0:2] == "TX":
            x.drop(["snap_CA","snap_WI"], axis=1, inplace=True)
        else:
            x.drop(["snap_CA","snap_TX"], axis=1, inplace=True)

        for name, cols in group_map.items():
            x = x.merge(lookups[name], on=cols, how="left")

        x = x.sort_values(["item_id", "Date"])
        g = x.groupby(["item_id"])["sales"]

        x["rolling_sold_mean_7"]  = g.transform(lambda s: s.shift(28).rolling(7).mean())
        x["rolling_sold_std_7"]   = g.transform(lambda s: s.shift(28).rolling(7).std())
        x["rolling_sold_mean_14"] = g.transform(lambda s: s.shift(28).rolling(14).mean())
        x["rolling_sold_std_14"]  = g.transform(lambda s: s.shift(28).rolling(14).std())
        x["rolling_sold_mean_28"] = g.transform(lambda s: s.shift(28).rolling(28).mean())
        x["rolling_sold_mean_60"] = g.transform(lambda s: s.shift(28).rolling(60).mean())

        for j in [1,2,7,14,30]:
            x[f"sales_lag_{j+28}"] = x.groupby(["item_id"])["sales"].shift(j+28)

        x["sellingTrend"] = x["sales_lag_29"] - x["item_id_enc_mean"]

        future_rows = x[x["Date"] > date].drop(columns=featurestodrop + ["sales"], errors="ignore")
        future_rows=save_mem(future_rows)
        future_rows.to_pickle(FUTURE_DIR + rf"\sales-{i}.pkl")

        del x, future_rows
        gc.collect()

    del combined
    print("Saved static future skeleton for all stores ->", FUTURE_DIR)
else:
    print(f"'sales' already has real data through day {future_days[-1]} — skipping skeleton build")

'sales' already has real data through day 1969 — skipping skeleton build


In [90]:
'''Baseline RMSE (CA_1): 2.0428

               feature  rmse_increase
      rolling_mean_1_7       1.099444
     rolling_mean_1_14       0.183153
      rolling_mean_7_7       0.103063
                  wday       0.099524
     rolling_mean_7_14       0.058326
state_item_id_enc_mean       0.047603
     rolling_mean_1_30       0.045471
              week_end       0.009001
    rolling_mean_14_14       0.007512
      dept_id_enc_mean       0.004661
     rolling_mean_7_30       0.004459
     rolling_mean_14_7       0.003089
          sales_lag_30       0.001943
             day_month       0.001844
    rolling_sold_std_7       0.001667
  rolling_sold_mean_14       0.001589
  rolling_sold_mean_60       0.001443
  rolling_sold_mean_28       0.001344
    rolling_mean_14_30       0.001316
            price_norm       0.001298
            sell_price       0.001141
     price_momentum_w1       0.000739
             price_max       0.000468
          sales_lag_58       0.000468
   rolling_sold_std_14       0.000432
          price_lag_w1       0.000308
            price_mean       0.000134
             price_min       0.000112
        cat_id_enc_std       0.000090
     price_momentum_m6       0.000084
          sellingTrend       0.000011
                 month       0.000007
                  Date       0.000000
                  year       0.000000
            week_month      -0.000036
          price_lag_m6      -0.000299
               snap_CA      -0.000344
 state_dept_id_enc_std      -0.001042
       item_id_enc_std      -0.001167
          sales_lag_42      -0.001961
          sales_lag_35      -0.002761
      item_id_enc_mean      -0.003474
          sales_lag_29      -0.003654'''


'Baseline RMSE (CA_1): 2.0428\n\n               feature  rmse_increase\n      rolling_mean_1_7       1.099444\n     rolling_mean_1_14       0.183153\n      rolling_mean_7_7       0.103063\n                  wday       0.099524\n     rolling_mean_7_14       0.058326\nstate_item_id_enc_mean       0.047603\n     rolling_mean_1_30       0.045471\n              week_end       0.009001\n    rolling_mean_14_14       0.007512\n      dept_id_enc_mean       0.004661\n     rolling_mean_7_30       0.004459\n     rolling_mean_14_7       0.003089\n          sales_lag_30       0.001943\n             day_month       0.001844\n    rolling_sold_std_7       0.001667\n  rolling_sold_mean_14       0.001589\n  rolling_sold_mean_60       0.001443\n  rolling_sold_mean_28       0.001344\n    rolling_mean_14_30       0.001316\n            price_norm       0.001298\n            sell_price       0.001141\n     price_momentum_w1       0.000739\n             price_max       0.000468\n          sales_lag_58       0.

In [91]:
import lightgbm as lgm
STORE = "CA_2"
perm_df = pd.read_pickle(OTHER + rf"\Sale-Store\sales-{STORE}.pkl")
feature_cols = [c for c in perm_df.columns if c not in ("sales", "item_id", "store_id")]
lgm_params={'subsample': 0.75,
             'num_leaves': 70, 
            'n_estimators': 500,
             'max_depth': 15,
             'max_bin': 100, 
            'learning_rate': 0.02, 
            'colsample_bytree': 0.7, 
            'objective': 'tweedie',
            'tweedie_variance_power': 1.1, 
            'boost_from_average': False,
              'random_state': 42,
                'n_jobs': -1, 
                'verbose': -1}
cols = ['dept_id_enc_mean','item_id_enc_mean','sales','state_id_enc_mean','cat_id_enc_mean','item_id_cat_id_enc_mean','dept_id_item_id_enc_mean','state_dept_id_enc_mean','state_item_id_enc_mean','state_item_dept_enc_mean','store_cat_id_enc_mean','store_dept_id_enc_mean','store_item_id_enc_mean','sales_lag_29','sales_lag_30','sales_lag_31','sales_lag_35','sales_lag_42','sales_lag_58','sellingTrend']
for col in cols:
    if col  in perm_df.columns:
        perm_df[col]=pd.to_numeric(perm_df[col])
perm_model = lgm.LGBMRegressor(**lgm_params)
perm_cutoff = 1913 - 28   # e.g. 1885 - 28 = 1857
perm_train = perm_df[perm_df["Date"] <= perm_cutoff]
perm_valid = perm_df[perm_df["Date"] > perm_cutoff]
perm_model.fit(perm_train[feature_cols], perm_train["sales"])
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))
base_preds = perm_model.predict(perm_valid[feature_cols])
base_score = rmse(perm_valid["sales"].values, base_preds)
print(f"Baseline RMSE ({STORE}): {base_score:.4f}\n")
results = []
for col in feature_cols:
    if perm_valid[col].dtype.name == "category":
        continue  # skip categoricals, shuffling can disrupt category codes
    shuffled = perm_valid.copy()
    shuffled[col] = np.random.permutation(shuffled[col].values)
    shuffled_preds = perm_model.predict(shuffled[feature_cols])
    shuffled_score = rmse(shuffled["sales"].values, shuffled_preds)
    results.append({"feature": col, "rmse_increase": shuffled_score - base_score})
perm_results = pd.DataFrame(results).sort_values("rmse_increase", ascending=False)
print(perm_results.to_string(index=False))

Baseline RMSE (CA_2): 1.9117

               feature  rmse_increase
      rolling_mean_1_7       0.787827
     rolling_mean_1_14       0.648277
state_item_id_enc_mean       0.151563
     rolling_mean_1_30       0.103530
                  wday       0.086872
      rolling_mean_7_7       0.077117
              week_end       0.020099
     rolling_mean_7_14       0.018093
          sales_lag_35       0.016568
          sales_lag_42       0.012637
     rolling_mean_7_30       0.011713
  rolling_sold_mean_60       0.004804
 state_dept_id_enc_std       0.004300
      item_id_enc_mean       0.003594
             price_max       0.002758
  rolling_sold_mean_28       0.002568
            sell_price       0.002239
      dept_id_enc_mean       0.002142
        cat_id_enc_std       0.001981
     price_momentum_w1       0.001647
          price_lag_m6       0.001638
       item_id_enc_std       0.001510
                 month       0.001315
             price_min       0.001106
            price_me